# Step 2: BERTopic Modeling

**입력**: `data/embeddings/` 의 최신 `_embeddings.npy` + `_metadata.csv`  
**출력**: `data/model_results/` 에 모델, 토픽 정보, 문서-토픽 매핑

---
**파이프라인 순서**  
1. (선택) 파라미터 튜닝 — UMAP/HDBSCAN 그리드 서치  
2. BERTopic 실행 — 토픽 추출 + 표현 정제

## 0. 환경 설정

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/korean-bertopic-pipeline'
else:
    PROJECT_DIR = str(Path('..').resolve())

sys.path.insert(0, PROJECT_DIR)
print(f'Project: {PROJECT_DIR}')

In [ ]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)')
else:
    print('CPU 모드')

## 1. 설정

In [ ]:
import os
from pipeline.config import load_config
from pipeline.embed import find_latest
from pipeline.tokenize import get_tokenizer

cfg = load_config(os.path.join(PROJECT_DIR, 'config.yaml'))
data_cfg  = cfg['data']
embed_cfg = cfg['embedding']
tok_cfg   = cfg['tokenizer']
model_cfg = cfg['model']

EMBED_DIR  = Path(PROJECT_DIR) / embed_cfg.get('output_dir', 'data/embeddings')
OUTPUT_DIR = Path(PROJECT_DIR) / model_cfg.get('output_dir', 'data/model_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'임베딩 경로: {EMBED_DIR}')
print(f'출력 경로:   {OUTPUT_DIR}')

## 2. 임베딩 + 메타데이터 로드

In [ ]:
import numpy as np
import pandas as pd

embed_file = find_latest(EMBED_DIR, '*_embeddings.npy')
meta_file  = find_latest(EMBED_DIR, '*_metadata.csv')

if not embed_file or not meta_file:
    raise FileNotFoundError('임베딩 파일 없음. 먼저 01_embed.ipynb 실행')

print(f'임베딩: {embed_file.name}')
embeddings = np.load(embed_file)
df = pd.read_csv(meta_file, encoding='utf-8-sig')

assert len(embeddings) == len(df), 'Shape mismatch!'
print(f'Shape: {embeddings.shape} | 문서: {len(df):,}')

## 3. 텍스트 전처리 (토크나이저)

In [ ]:
TEXT_COL = data_cfg.get('text_column', 'text')
for col in [TEXT_COL, 'text', 'abstract', 'content', '본문']:
    if col in df.columns:
        TEXT_COL = col
        break

docs_raw = df[TEXT_COL].fillna('').tolist()
print(f'텍스트 컬럼: {TEXT_COL!r} | {len(docs_raw):,}건')

In [ ]:
# 토크나이저 초기화 (config.yaml의 tokenizer.type 사용)
# type: 'kiwi' (한국어), 'whitespace' (범용), 'none' (BERTopic 기본)

tok_type = tok_cfg.get('type', 'whitespace')
ud_path  = tok_cfg.get('user_dict_path')
sw_path  = tok_cfg.get('stopwords_path')
min_len  = tok_cfg.get('min_token_len', 2)

if ud_path:
    ud_path = os.path.join(PROJECT_DIR, ud_path)
if sw_path:
    sw_path = os.path.join(PROJECT_DIR, sw_path)

tokenizer = get_tokenizer(tok_type, ud_path, sw_path, min_len)
print(f'토크나이저: {tok_type}')

In [ ]:
from tqdm import tqdm

if hasattr(tokenizer, 'tokenize_batch'):
    docs_tokenized = tokenizer.tokenize_batch(docs_raw)
else:
    docs_tokenized = [' '.join(tokenizer.tokenize(t)) for t in tqdm(docs_raw, desc='Tokenizing')]

# 샘플 확인
print('\n샘플 (원문):')
print(docs_raw[0][:100])
print('\n샘플 (토큰화):')
print(docs_tokenized[0][:100])

## 4. (선택) 파라미터 튜닝

이미 튜닝 결과(`*_tuned_config.json`)가 있으면 건너뛰세요.

In [ ]:
# 튜닝 결과 확인
tuned_config_file = find_latest(EMBED_DIR, '*_tuned_config.json')
if tuned_config_file:
    print(f'기존 튜닝 결과 발견: {tuned_config_file.name}')
    print('→ 섹션 5로 건너뛰세요')
else:
    print('튜닝 결과 없음 → 아래 셀에서 그리드 서치 실행 또는 기본값 사용')

In [ ]:
# 미니 그리드 서치 (빠른 탐색, ~10-20분)
# 전체 그리드는 scripts/02_tune.py 실행 권장

RUN_MINI_TUNE = False  # True로 변경 시 실행

if RUN_MINI_TUNE and tuned_config_file is None:
    import json, warnings
    from itertools import product
    from umap import UMAP
    from hdbscan import HDBSCAN
    from bertopic import BERTopic
    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.metrics import silhouette_score
    from pipeline.metrics import calculate_diversity, calculate_coherence, composite_score
    warnings.filterwarnings('ignore')

    MINI_GRID = {
        'umap_n_neighbors': [10, 15],
        'umap_n_components': [5, 10],
        'hdbscan_min_cluster_size': [100, 150, 200],
        'hdbscan_min_samples': [10, 15],
    }
    TARGET = tuple(cfg['tuning'].get('target_topics', [20, 50]))

    results = []
    combos = list(product(
        MINI_GRID['umap_n_neighbors'], MINI_GRID['umap_n_components'],
        MINI_GRID['hdbscan_min_cluster_size'], MINI_GRID['hdbscan_min_samples'],
    ))
    print(f'미니 그리드: {len(combos)} trials')

    prev_umap = None
    reduced = None

    for nb, nc, mcs, ms in tqdm(combos):
        try:
            if (nb, nc) != prev_umap:
                umap_m = UMAP(n_neighbors=nb, n_components=nc, min_dist=0.0, metric='cosine', random_state=42, n_jobs=1)
                reduced = umap_m.fit_transform(embeddings)
                prev_umap = (nb, nc)

            hdb = HDBSCAN(min_cluster_size=mcs, min_samples=ms, cluster_selection_method='eom', metric='euclidean', prediction_data=True)
            labels = hdb.fit_predict(reduced)
            n_cl = len(set(labels)) - (1 if -1 in labels else 0)
            if n_cl < 2: continue

            noise = float((labels == -1).sum() / len(labels))
            nn = labels != -1
            sil = float(silhouette_score(reduced[nn], labels[nn]) if nn.sum() > n_cl else 0.0)

            vc = CountVectorizer(tokenizer=lambda x: x.split(), token_pattern=None, min_df=3, max_df=0.95)
            tm = BERTopic(umap_model=umap_m, hdbscan_model=hdb, vectorizer_model=vc, embedding_model=None, verbose=False)
            tm.fit(docs_tokenized, embeddings)
            td = tm.get_topics()
            dws = [set(d.split()) for d in docs_tokenized if d.strip()]

            div = calculate_diversity(td)
            coh = calculate_coherence(td, dws)
            comp = composite_score(sil, noise, div, coh, n_cl, TARGET)
            results.append({'n_neighbors': nb, 'n_components': nc, 'min_cluster_size': mcs, 'min_samples': ms,
                            'n_topics': n_cl, 'noise_ratio': round(noise, 4), 'silhouette': round(sil, 4),
                            'diversity': round(div, 4), 'coherence': round(coh, 4), 'composite': round(comp, 4)})
        except Exception:
            continue

    results_df = pd.DataFrame(results).sort_values('composite', ascending=False)
    best = results_df.iloc[0]
    ts = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')

    tuned_out = {
        'timestamp': ts,
        'umap': {'n_neighbors': int(best.n_neighbors), 'n_components': int(best.n_components), 'min_dist': 0.0, 'metric': 'cosine'},
        'hdbscan': {'min_cluster_size': int(best.min_cluster_size), 'min_samples': int(best.min_samples), 'cluster_selection_method': 'eom', 'metric': 'euclidean'},
        'best_metrics': {'n_topics': int(best.n_topics), 'composite': float(best.composite)},
    }
    config_path = EMBED_DIR / f'{ts}_tuned_config.json'
    with open(config_path, 'w') as f:
        json.dump(tuned_out, f, indent=2)

    print(f'\n최적: topics={int(best.n_topics)}, composite={float(best.composite):.4f}')
    print(f'저장: {config_path}')
    tuned_config_file = config_path
    display(results_df.head(10))

## 5. BERTopic 모델 실행

In [ ]:
import json, time, logging
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

# 튜닝 결과 로드 (없으면 기본값)
DEFAULT_UMAP    = {'n_neighbors': 15, 'n_components': 10, 'min_dist': 0.0, 'metric': 'cosine'}
DEFAULT_HDBSCAN = {'min_cluster_size': 150, 'min_samples': 15, 'cluster_selection_method': 'eom'}

tuned = None
if tuned_config_file and Path(tuned_config_file).exists():
    with open(tuned_config_file) as f:
        tuned = json.load(f)

umap_p   = tuned['umap']    if tuned else DEFAULT_UMAP
hdbscan_p = tuned['hdbscan'] if tuned else DEFAULT_HDBSCAN

print(f'UMAP:    {umap_p}')
print(f'HDBSCAN: {hdbscan_p}')

In [ ]:
umap_model = UMAP(
    n_neighbors=umap_p['n_neighbors'], n_components=umap_p['n_components'],
    min_dist=umap_p['min_dist'], metric=umap_p.get('metric', 'cosine'),
    random_state=42, n_jobs=1,
)
hdbscan_model = HDBSCAN(
    min_cluster_size=hdbscan_p['min_cluster_size'],
    min_samples=hdbscan_p.get('min_samples', 15),
    cluster_selection_method=hdbscan_p.get('cluster_selection_method', 'eom'),
    metric='euclidean', prediction_data=True,
)
vectorizer_model = CountVectorizer(
    tokenizer=lambda x: x.split(), token_pattern=None,
    min_df=model_cfg.get('min_df', 5), max_df=model_cfg.get('max_df', 0.95),
)
representation_model = {
    'KeyBERT': KeyBERTInspired(),
    'MMR': MaximalMarginalRelevance(diversity=model_cfg.get('mmr_diversity', 0.3)),
}
embedding_model = SentenceTransformer(embed_cfg.get('model', 'BAAI/bge-m3'))

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,
    calculate_probabilities=False,
    verbose=True,
)
print('BERTopic 초기화 완료')

In [ ]:
print(f'BERTopic 실행 ({len(docs_tokenized):,} 문서)...')
t0 = time.time()
topics, _ = topic_model.fit_transform(docs_tokenized, embeddings)
elapsed = time.time() - t0
print(f'완료: {elapsed:.0f}초 ({elapsed/60:.1f}분)')

## 6. 결과 확인

In [ ]:
topic_info = topic_model.get_topic_info()
n_topics   = int((topic_info['Topic'] != -1).sum())
n_outliers = int(topic_info.loc[topic_info['Topic'] == -1, 'Count'].sum()) if -1 in topic_info['Topic'].values else 0

print(f'토픽 수: {n_topics} | 아웃라이어: {n_outliers:,} ({n_outliers/len(topics)*100:.1f}%)')
display(topic_info.head(15))

## 7. 저장

In [ ]:
ts = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')

# 모델 저장
model_path = OUTPUT_DIR / f'{ts}_bertopic_model'
topic_model.save(str(model_path), serialization='safetensors', save_ctfidf=True)

# 토픽 정보
topic_info.to_csv(OUTPUT_DIR / f'{ts}_topic_info.csv', index=False, encoding='utf-8-sig')

# 문서-토픽 매핑
df['topic'] = topics
df.to_csv(OUTPUT_DIR / f'{ts}_doc_topics.csv', index=False, encoding='utf-8-sig')

# 실행 설정
run_cfg = {
    'timestamp': ts, 'n_documents': len(topics),
    'n_topics': n_topics, 'n_outliers': n_outliers,
    'elapsed_seconds': round(elapsed, 1),
    'embedding_model': embed_cfg.get('model'),
    'tokenizer': tok_cfg.get('type'),
    'umap_params': umap_p, 'hdbscan_params': hdbscan_p,
}
with open(OUTPUT_DIR / f'{ts}_run_config.json', 'w') as f:
    json.dump(run_cfg, f, ensure_ascii=False, indent=2)

print(f'모델: {model_path}')
print(f'출력: {OUTPUT_DIR}')
print('완료 ✓')